In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = pd.read_csv("data/processed/final_feature_matrix.csv", index_col=0, parse_dates=True)
stationary_vars = df[["d_log_eua", "ttf_gas", "coal_price", "renewable_share"]].diff().dropna()

split = int(len(stationary_vars) * 0.80)
train_var, test_var = stationary_vars.iloc[:split], stationary_vars.iloc[split:]

# 1. VAR Model
var_model = VAR(train_var)
var_results = var_model.fit(maxlags=5, ic="aic")
print(var_results.summary())

# 2. ARIMAX Model
y_train = stationary_vars["d_log_eua"].iloc[:split]
X_train = stationary_vars[["ttf_gas", "coal_price", "renewable_share"]].iloc[:split]
X_test  = stationary_vars[["ttf_gas", "coal_price", "renewable_share"]].iloc[split:]

arimax = SARIMAX(y_train, exog=X_train, order=(2, 0, 1)).fit(disp=False)
print(arimax.summary())

# Save predictions for final evaluation
arimax_preds = arimax.forecast(steps=len(test_var), exog=X_test)
arimax_preds.to_csv("data/processed/baseline_arimax_preds.csv")

/Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting/venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting/venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimizat

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Wed, 17, Jun, 2026
Time:                     16:36:32
--------------------------------------------------------------------
No. of Equations:         4.00000    BIC:                    3.67077
Nobs:                     1020.00    HQIC:                   3.41906
Log likelihood:          -7370.41    FPE:                    26.1801
AIC:                      3.26498    Det(Omega_mle):         24.1306
--------------------------------------------------------------------
Results for equation d_log_eua
                        coefficient       std. error           t-stat            prob
-------------------------------------------------------------------------------------
const                     -0.000186         0.005500           -0.034           0.973
L1.d_log_eua              -1.395794         0.030232          -46.169           0.000
L1.ttf_gas                 0.000489 